# Exercise — Encode Governance Obligations in a Data Contract (Solution)

**Trailhead Provisions** has an ungoverned `orders` contract: it describes columns but carries no
policy. Extend it with a **governance block** so the validator passes both structural schema
validation and semantic governance checks, then explain how a policy change propagates across
producers and downstream consumers. See `INSTRUCTIONS.md`.

In [1]:
import seed_data
from governance_toolkit import validate_contract, build_lineage

orders_meta = seed_data._build_catalog().table("orders")
print(f"Catalog metadata for 'orders' -> domain={orders_meta.domain!r}, retention={orders_meta.retention!r}")

contract = {
    "name": "orders",
    "domain": "E-commerce Orders",
    "owner": "orders-team@trailhead.example",
    "version": "1.0.0",
    "columns": [
        {"name": "order_id", "type": "integer"},
        {"name": "customer_id", "type": "integer"},
        {"name": "order_ts", "type": "timestamp"},
        {"name": "status", "type": "string"},
        {"name": "amount", "type": "double"},
        {"name": "currency", "type": "string"},
    ],
}
ok, errs = validate_contract(contract)
print("Base contract valid?", ok, "| errors:", errs)   # provided: fails, no governance block

Catalog metadata for 'orders' -> domain='E-commerce Orders', retention='P7Y'
Base contract valid? False | errors: ["(root): 'governance' is a required property"]


## 1. Add the governance block
Add `pii_tags`, `retention` (look up the `orders` retention period from the catalog metadata printed above or specify an ISO-8601 duration matching `^P\d+[YMD]$`), and `quality_slos` (list of `{dimension, threshold}` where `0 < threshold <= 1`), and mark PII columns with `col["pii"] = True`. Make `validate_contract` pass both JSON Schema and semantic cross-checks.

In [2]:
for col in contract["columns"]:
    col["pii"] = col["name"] in ("customer_id",)   # FK to a PII entity
contract["governance"] = {
    "pii_tags": ["customer_id"],
    "retention": "P7Y",
    "quality_slos": [
        {"dimension": "completeness", "threshold": 0.99},
        {"dimension": "validity", "threshold": 0.98},
        {"dimension": "accuracy", "threshold": 0.999},
    ],
}
ok, errs = validate_contract(contract)
print("Extended contract valid?", ok, "| errors:", errs)
assert ok, "fix the governance block until the validator passes"
contract["governance"]

Extended contract valid? True | errors: []


{'pii_tags': ['customer_id'], 'retention': 'P7Y', 'quality_slos': [{'dimension': 'completeness', 'threshold': 0.99}, {'dimension': 'validity', 'threshold': 0.98}, {'dimension': 'accuracy', 'threshold': 0.999}]}

In [3]:
# Provided: inspect the downstream consumers of `orders` before writing your propagation analysis.
lin = build_lineage()
print("Downstream of 'orders':", lin.trace_downstream("orders"))
lin.events()

Downstream of 'orders': ['order_carbon', 'marketing_audience', 'quarterly_sustainability']


,job,inputs,outputs
0,build_order_carbon,"orders, shipment",order_carbon
1,build_quarterly_sustainability,order_carbon,quarterly_sustainability
2,build_customer_360,"customer, campaign_membership",customer_360
3,build_marketing_audience,"customer_360, orders",marketing_audience


## 2. Policy-propagation write-up
Replace the cell below with your write-up.

### Policy-propagation write-up

Adding `retention: "P7Y"`, `pii_tags: ["customer_id"]`, and `quality_slos` to the `orders` contract is not a local change — it propagates across both the producing pipeline and every downstream dataset returned by `trace_downstream("orders")` (`order_carbon`, `quarterly_sustainability`, and `marketing_audience`):

- **Producers (E-commerce Orders pipeline):** must now validate records against the declared quality SLOs (such as `amount >= 0` accuracy `>= 99.9%`) before publishing, tag `customer_id` as `PII` in Glue, and enforce a 7-year (`P7Y`) retention and deletion lifecycle.
- **Downstream consumer — `marketing_audience` (PII inheritance):** the `build_marketing_audience` job joins `orders` (`customer_id`) with `customer_360`. Because `orders.customer_id` is declared `PII` (a linkable customer identifier), `marketing_audience` inherits the PII obligation and must classify, mask, and honor GDPR subject-erasure and consent rules on `customer_id`.
- **Downstream consumers — `order_carbon` and `quarterly_sustainability` (CSRD retention dependency):** the `build_order_carbon` job joins `orders` (`order_id`, `order_ts`, `status`) with `shipment` without reading `customer_id`, so neither `order_carbon` nor `quarterly_sustainability` (the CSRD carbon disclosure report) inherits PII. **However**, `orders`'s `P7Y` retention governs how many years of historical quarters remain auditable for CSRD sustainability disclosures — deleting `orders` rows early would break historical traceability for `order_carbon` and `quarterly_sustainability`.
- **Propagation mechanism:** adding or changing the governance block **version-bumps** the `orders` contract (`1.0.0` → `1.1.0`). Every downstream contract referencing `orders@1.0.0` is automatically flagged for review, and the central publish gate refuses to publish a downstream product referencing an outdated upstream contract version. Remediation proceeds **upstream-first** in topological order: update and validate `orders` first, then update `order_carbon` → `quarterly_sustainability` and `marketing_audience`.

---
## Solution Summary & Common Mistakes to Avoid

### Key points
- **Two-layer contract validation:** `validate_contract()` enforces both JSON Schema structure (`CONTRACT_SCHEMA`) and semantic rules (`col["pii"] = True` must match `governance["pii_tags"]`, and every SLO `threshold` must be in `(0, 1]`).
- **True downstream consumers of `orders` (`trace_downstream("orders")`):**
  - `marketing_audience` joins `orders.customer_id` with `customer_360` and therefore **inherits the PII obligation**.
  - `order_carbon` and `quarterly_sustainability` read `orders` (`order_id`, `order_ts`, `status`) without `customer_id` (no PII inheritance), but **depend on `orders`'s `P7Y` retention** so historical CSRD carbon disclosures remain auditable.
- **Propagation mechanism:** semantic version-bump (`1.0.0` → `1.1.0`) + central publish gate + upstream-first topological remediation.

### Common Mistakes to Avoid
- **An invalid `retention` format** — must match ISO-8601 `^P\d+[YMD]$` (e.g., `P7Y`), not `"7 years"`.
- **Mismatch between `col["pii"]` and `governance["pii_tags"]`** — marking `pii_tags: ["customer_id"]` without setting `col["pii"] = True` on the `customer_id` column fails semantic validation.
- **Supplying percentage numbers (`99.9`) instead of proportions (`0.999`)** for SLO `threshold`.
- **Claiming `customer_360` consumes `orders`** — `customer_360` is built from `customer` and `campaign_membership`; `orders` joins with `customer_360` downstream in `marketing_audience`.